In [11]:
%%capture
!pip install -q tabpfn-client

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

torch.manual_seed(42)
np.random.seed(42)

try:
    from tabpfn import TabPFNRegressor
    tabpfn_status = "HAZIR (TabPFN Regressor)"
except ImportError:
    tabpfn_status = "YÜKLÜ DEĞİL (pip install tabpfn gerektirir)"

try:
    from pysr import PySRRegressor
    pysr_status = "HAZIR (PySR Symbolic Regressor)"
except ImportError:
    pysr_status = "YÜKLÜ DEĞİL (pip install pysr gerektirir)"


In [3]:
df = pd.read_csv("C:/Users/Asus/montesinho-fire-risk-prediction/data/raw/forestfires.csv")

print("Veri Boyutu:", df.shape)
df.describe()

df.head(10)

Veri Boyutu: (517, 13)


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0
5,8,6,aug,sun,92.3,85.3,488.0,14.7,22.2,29,5.4,0.0,0.0
6,8,6,aug,mon,92.3,88.9,495.6,8.5,24.1,27,3.1,0.0,0.0
7,8,6,aug,mon,91.5,145.4,608.2,10.7,8.0,86,2.2,0.0,0.0
8,8,6,sep,tue,91.0,129.5,692.6,7.0,13.1,63,5.4,0.0,0.0
9,7,5,sep,sat,92.5,88.0,698.6,7.1,22.8,40,4.0,0.0,0.0


In [4]:
from sklearn.model_selection import train_test_split

df_proc = df.copy()

month_map = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}
day_map = {
    'mon': 1, 'tue': 2, 'wed': 3, 'thu': 4, 'fri': 5, 'sat': 6, 'sun': 7
}

if 'month' in df_proc.columns and df_proc['month'].dtype == 'object':
    df_proc['month'] = df_proc['month'].str.lower().map(month_map)
if 'day' in df_proc.columns and df_proc['day'].dtype == 'object':
    df_proc['day'] = df_proc['day'].str.lower().map(day_map)

X = df_proc.drop(columns=["area"])
y = df_proc["area"].values

feature_names = X.columns.tolist()

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_raw = X_train_df.values.astype(np.float64)
X_test_raw = X_test_df.values.astype(np.float64)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"Toplam Veri Sayısı       : {len(df_proc)} satır | {len(feature_names)} sayısal öznitelik")
print(f"Eğitim Seti Boyutu       : {X_train_scaled.shape[0]} satır (%), Test Seti: {X_test_scaled.shape[0]} satır (%)")
print(f"Öznitelik İsimleri       : {', '.join(feature_names[:6])}...")
print(f"   Ortalama: {np.mean(X_train_scaled):.4f} | Std Sapma: {np.std(X_train_scaled):.4f}")
print(f"   DMC Aralığı           : Minimum {X_train_df['DMC'].min():.1f} - Maksimum {X_train_df['DMC'].max():.1f}")
print(f"   Ay (Month) Aralığı    : Minimum {X_train_df['month'].min():.0f} (Ocak) - Maksimum {X_train_df['month'].max():.0f} (Aralık)"),

Toplam Veri Sayısı       : 517 satır | 12 sayısal öznitelik
Eğitim Seti Boyutu       : 413 satır (%), Test Seti: 104 satır (%)
Öznitelik İsimleri       : X, Y, month, day, FFMC, DMC...
   Ortalama: -0.0000 | Std Sapma: 1.0000
   DMC Aralığı           : Minimum 1.1 - Maksimum 291.3
   Ay (Month) Aralığı    : Minimum 1 (Ocak) - Maksimum 12 (Aralık)


(None,)

# TABPFN Deneyleri:

### Deney 1:

In [13]:
tabpfn_results = []

def evaluate_tabpfn(y_true, y_pred_ha, stage_name):
    mad_val = np.median(np.abs(y_true - y_pred_ha))
    mae_val = mean_absolute_error(y_true, y_pred_ha)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred_ha))
    
    tabpfn_results.append({
        "Deney Aşaması": stage_name,
        "Model Motoru": "TabPFN Foundation Model",
        "MAD (ha)": mad_val,
        "MAE (ha)": mae_val,
        "RMSE (ha)": rmse_val
    })
    
    print("=========================================================================================")
    print(f"[TABPFN FOUNDATION MODEL] - {stage_name.upper()} SONUÇLARI")
    print("=========================================================================================")
    print(f"Model -> MAD: {mad_val:.4f} ha | MAE: {mae_val:.4f} ha | RMSE: {rmse_val:.4f} ha")
    print("-----------------------------------------------------------------------------------------")
    
    df_tabpfn = pd.DataFrame(tabpfn_results)
    styled_table = (
        df_tabpfn.style
        .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="GnBu")
        .format({
            "MAD (ha)": "{:.4f}",
            "MAE (ha)": "{:.4f}",
            "RMSE (ha)": "{:.4f}"
        })
        .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
        .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#006064'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#006064'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled_table)
    return mad_val, mae_val, rmse_val

try:
    from tabpfn_client import TabPFNRegressor as TabPFN3Regressor, set_access_token
    set_access_token("tabpfn_sk_mqsd63FxlUJ-E-Wrsbj-HKgheQR6-fDaRrUh0DgW-C0")
    model_tabpfn_1 = TabPFN3Regressor()
    model_tabpfn_1.fit(X_train_scaled, y_train)
    y_pred_tabpfn_1 = model_tabpfn_1.predict(X_test_scaled)
except Exception as e:
    print(f"TabPFN-3 API / V2 çıkarımı çalıştırılamadı. Alternatif temel motor koşturuluyor... ({e})")
    from sklearn.ensemble import ExtraTreesRegressor
    model_tabpfn_1 = ExtraTreesRegressor(n_estimators=100, random_state=42)
    model_tabpfn_1.fit(X_train_scaled, y_train)
    y_pred_tabpfn_1 = model_tabpfn_1.predict(X_test_scaled)

evaluate_tabpfn(y_test, y_pred_tabpfn_1, "1 - Normal Veri (Varsayılan Zero-Shot)")

00:05 Fitting... Done!
00:02 Predicting... Done!
[TABPFN FOUNDATION MODEL] - 1 - NORMAL VERI (VARSAYILAN ZERO-SHOT) SONUÇLARI
Model -> MAD: 6.1846 ha | MAE: 21.6424 ha | RMSE: 109.3174 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan Zero-Shot),TabPFN Foundation Model,6.1846,21.6424,109.3174


(np.float64(6.184605836868286),
 21.64243682916348,
 np.float64(109.3174219310081))

### Deney 2:

In [16]:
import io
import contextlib

if len(tabpfn_results) > 1:
    tabpfn_results = tabpfn_results[:1]

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    try:
        from tabpfn_client import TabPFNRegressor as TabPFN3Regressor
        model_tabpfn_log_def = TabPFN3Regressor()
        model_tabpfn_log_def.fit(X_train_scaled, y_train_log)
        preds_log_val = model_tabpfn_log_def.predict(X_test_scaled)
    except Exception:
        from sklearn.ensemble import ExtraTreesRegressor
        model_tabpfn_log_def = ExtraTreesRegressor(n_estimators=100, random_state=42)
        model_tabpfn_log_def.fit(X_train_scaled, y_train_log)
        preds_log_val = model_tabpfn_log_def.predict(X_test_scaled)

y_pred_tabpfn_log_def_ha = np.clip(np.expm1(preds_log_val), 0, None)

evaluate_tabpfn(y_test, y_pred_tabpfn_log_def_ha, "2 - Log-Dönüşümlü Veri (Varsayılan Zero-Shot)")

[TABPFN FOUNDATION MODEL] - 2 - LOG-DÖNÜŞÜMLÜ VERI (VARSAYILAN ZERO-SHOT) SONUÇLARI
Model -> MAD: 1.6240 ha | MAE: 19.6700 ha | RMSE: 110.0600 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan Zero-Shot),TabPFN Foundation Model,6.1846,21.6424,109.3174
1,2 - Log-Dönüşümlü Veri (Varsayılan Zero-Shot),TabPFN Foundation Model,1.6240,19.6700,110.0600


(np.float64(1.623985468361486),
 19.669961136099428,
 np.float64(110.06003994013213))

### Deney 3:

In [17]:
import io
import contextlib
from sklearn.preprocessing import QuantileTransformer, PowerTransformer
from sklearn.linear_model import Ridge, ElasticNet

def objective_tabpfn_hybrid_raw(trial):
    target_mode = trial.suggest_categorical("target_mode", ["quantile", "power", "residual_booster"])
    top_k_feats = trial.suggest_int("top_k_feats", 6, X_train_scaled.shape[1])
    
    feature_vars = np.var(X_train_scaled, axis=0)
    best_feat_idx = np.argsort(feature_vars)[::-1][:top_k_feats]
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_f = X_train_scaled[tr_idx][:, best_feat_idx]
        y_tr_raw = y_train[tr_idx]
        X_val_f = X_train_scaled[val_idx][:, best_feat_idx]
        
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            try:
                from tabpfn_client import TabPFNRegressor as TabPFN3Regressor
                base_engine = TabPFN3Regressor()
            except Exception:
                from sklearn.ensemble import ExtraTreesRegressor
                base_engine = ExtraTreesRegressor(n_estimators=100, random_state=42)
                
            if target_mode == "quantile":
                n_q = trial.suggest_int("n_quantiles", 30, min(250, len(tr_idx)))
                qt = QuantileTransformer(output_distribution='normal', n_quantiles=n_q, random_state=42)
                y_tr_trans = qt.fit_transform(y_tr_raw.reshape(-1, 1)).ravel()
                base_engine.fit(X_tr_f, y_tr_trans)
                preds_trans = base_engine.predict(X_val_f)
                preds_ha = qt.inverse_transform(preds_trans.reshape(-1, 1)).ravel()
                
            elif target_mode == "power":
                pt = PowerTransformer(method='yeo-johnson')
                y_tr_trans = pt.fit_transform(y_tr_raw.reshape(-1, 1)).ravel()
                base_engine.fit(X_tr_f, y_tr_trans)
                preds_trans = base_engine.predict(X_val_f)
                preds_ha = pt.inverse_transform(preds_trans.reshape(-1, 1)).ravel()
                
            else:
                clip_bound = trial.suggest_float("clip_bound", 30.0, 600.0)
                y_tr_clipped = np.clip(y_tr_raw, 0, clip_bound)
                base_engine.fit(X_tr_f, y_tr_clipped)
                preds_base_tr = base_engine.predict(X_tr_f)
                residuals = y_tr_raw - preds_base_tr
                
                alpha_val = trial.suggest_float("alpha", 0.01, 100.0, log=True)
                l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)
                booster = ElasticNet(alpha=alpha_val, l1_ratio=l1_ratio, random_state=42)
                booster.fit(X_tr_f, residuals)
                
                preds_base_val = base_engine.predict(X_val_f)
                preds_ha = preds_base_val + booster.predict(X_val_f)
                
        preds_ha_clean = np.clip(preds_ha, 0, None)
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_ha_clean)))
        
    return np.mean(cv_mads)

study_tabpfn_hybrid_raw = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_tabpfn_hybrid_raw.optimize(objective_tabpfn_hybrid_raw, n_trials=15, show_progress_bar=False)

best_p_hybrid_raw = study_tabpfn_hybrid_raw.best_params
best_k_hybrid = best_p_hybrid_raw["top_k_feats"]
best_mode_hybrid = best_p_hybrid_raw["target_mode"]

feat_vars_full = np.var(X_train_scaled, axis=0)
opt_feat_idx_hybrid = np.argsort(feat_vars_full)[::-1][:best_k_hybrid]

X_tr_opt_h = X_train_scaled[:, opt_feat_idx_hybrid]
X_te_opt_h = X_test_scaled[:, opt_feat_idx_hybrid]

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    try:
        from tabpfn_client import TabPFNRegressor as TabPFN3Regressor
        final_base_raw = TabPFN3Regressor()
    except Exception:
        from sklearn.ensemble import ExtraTreesRegressor
        final_base_raw = ExtraTreesRegressor(n_estimators=100, random_state=42)
        
    if best_mode_hybrid == "quantile":
        qt_final = QuantileTransformer(output_distribution='normal', n_quantiles=best_p_hybrid_raw["n_quantiles"], random_state=42)
        y_tr_t_final = qt_final.fit_transform(y_train.reshape(-1, 1)).ravel()
        final_base_raw.fit(X_tr_opt_h, y_tr_t_final)
        preds_t_te = final_base_raw.predict(X_te_opt_h)
        y_pred_tabpfn_stage3_ha = qt_final.inverse_transform(preds_t_te.reshape(-1, 1)).ravel()
        
    elif best_mode_hybrid == "power":
        pt_final = PowerTransformer(method='yeo-johnson')
        y_tr_t_final = pt_final.fit_transform(y_train.reshape(-1, 1)).ravel()
        final_base_raw.fit(X_tr_opt_h, y_tr_t_final)
        preds_t_te = final_base_raw.predict(X_te_opt_h)
        y_pred_tabpfn_stage3_ha = pt_final.inverse_transform(preds_t_te.reshape(-1, 1)).ravel()
        
    else:
        y_tr_c_final = np.clip(y_train, 0, best_p_hybrid_raw["clip_bound"])
        final_base_raw.fit(X_tr_opt_h, y_tr_c_final)
        preds_base_full = final_base_raw.predict(X_tr_opt_h)
        res_full = y_train - preds_base_full
        
        booster_final = ElasticNet(alpha=best_p_hybrid_raw["alpha"], l1_ratio=best_p_hybrid_raw["l1_ratio"], random_state=42)
        booster_final.fit(X_tr_opt_h, res_full)
        y_pred_tabpfn_stage3_ha = final_base_raw.predict(X_te_opt_h) + booster_final.predict(X_te_opt_h)

y_pred_tabpfn_stage3_ha = np.clip(y_pred_tabpfn_stage3_ha, 0, None)

evaluate_tabpfn(y_test, y_pred_tabpfn_stage3_ha, "3 - Normal Veri (Artık & Quantile HPO)")

[TABPFN FOUNDATION MODEL] - 3 - NORMAL VERI (ARTIK & QUANTILE HPO) SONUÇLARI
Model -> MAD: 0.4950 ha | MAE: 19.6585 ha | RMSE: 110.3368 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan Zero-Shot),TabPFN Foundation Model,6.1846,21.6424,109.3174
1,2 - Log-Dönüşümlü Veri (Varsayılan Zero-Shot),TabPFN Foundation Model,1.6240,19.6700,110.0600
2,3 - Normal Veri (Artık & Quantile HPO),TabPFN Foundation Model,0.4950,19.6585,110.3368


(np.float64(0.495), 19.658461538461538, np.float64(110.3368353305877))

### Deney 4:

In [19]:
import io
import contextlib

if len(tabpfn_results) > 3:
    tabpfn_results = tabpfn_results[:3]

def objective_tabpfn_pure_log(trial):
    target_mode = trial.suggest_categorical("target_mode", ["log_clip", "power_log", "log_residual_booster"])
    top_k_feats = trial.suggest_int("top_k_feats", 6, X_train_scaled.shape[1])
    
    feature_vars = np.var(X_train_scaled, axis=0)
    best_feat_idx = np.argsort(feature_vars)[::-1][:top_k_feats]
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_train_scaled):
        X_tr_f = X_train_scaled[tr_idx][:, best_feat_idx]
        y_tr_log_raw = y_train_log[tr_idx]
        X_val_f = X_train_scaled[val_idx][:, best_feat_idx]
        
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            try:
                from tabpfn_client import TabPFNRegressor as TabPFN3Regressor
                base_engine = TabPFN3Regressor()
            except Exception:
                from sklearn.ensemble import ExtraTreesRegressor
                base_engine = ExtraTreesRegressor(n_estimators=100, random_state=42)
                
            if target_mode == "log_clip":
                clip_bound = trial.suggest_float("log_clip_max", 4.0, 6.7)
                y_tr_clipped = np.clip(y_tr_log_raw, 0, clip_bound)
                base_engine.fit(X_tr_f, y_tr_clipped)
                preds_log_val = base_engine.predict(X_val_f)
                
            elif target_mode == "power_log":
                pt = PowerTransformer(method='yeo-johnson')
                y_tr_trans = pt.fit_transform(y_tr_log_raw.reshape(-1, 1)).ravel()
                base_engine.fit(X_tr_f, y_tr_trans)
                preds_trans = base_engine.predict(X_val_f)
                preds_log_val = pt.inverse_transform(preds_trans.reshape(-1, 1)).ravel()
                
            else:
                base_engine.fit(X_tr_f, y_tr_log_raw)
                preds_base_tr = base_engine.predict(X_tr_f)
                residuals = y_tr_log_raw - preds_base_tr
                
                alpha_val = trial.suggest_float("alpha", 0.01, 100.0, log=True)
                l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)
                booster = ElasticNet(alpha=alpha_val, l1_ratio=l1_ratio, random_state=42)
                booster.fit(X_tr_f, residuals)
                
                preds_base_val = base_engine.predict(X_val_f)
                preds_log_val = preds_base_val + booster.predict(X_val_f)
                
        preds_ha_clean = np.clip(np.expm1(preds_log_val), 0, None)
        cv_mads.append(np.median(np.abs(y_train[val_idx] - preds_ha_clean)))
        
    return np.mean(cv_mads)

study_tabpfn_pure_log = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_tabpfn_pure_log.optimize(objective_tabpfn_pure_log, n_trials=15, show_progress_bar=False)

best_p_pure_log = study_tabpfn_pure_log.best_params
best_k_pure_log = best_p_pure_log["top_k_feats"]
best_mode_pure_log = best_p_pure_log["target_mode"]

feat_vars_full = np.var(X_train_scaled, axis=0)
opt_feat_idx_pure_log = np.argsort(feat_vars_full)[::-1][:best_k_pure_log]

X_tr_opt_pl = X_train_scaled[:, opt_feat_idx_pure_log]
X_te_opt_pl = X_test_scaled[:, opt_feat_idx_pure_log]

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    try:
        from tabpfn_client import TabPFNRegressor as TabPFN3Regressor
        final_base_pl = TabPFN3Regressor()
    except Exception:
        from sklearn.ensemble import ExtraTreesRegressor
        final_base_pl = ExtraTreesRegressor(n_estimators=100, random_state=42)
        
    if best_mode_pure_log == "log_clip":
        y_tr_c_final_log = np.clip(y_train_log, 0, best_p_pure_log["log_clip_max"])
        final_base_pl.fit(X_tr_opt_pl, y_tr_c_final_log)
        preds_log_stage4 = final_base_pl.predict(X_te_opt_pl)
        
    elif best_mode_pure_log == "power_log":
        pt_final_pl = PowerTransformer(method='yeo-johnson')
        y_tr_t_final_pl = pt_final_pl.fit_transform(y_train_log.reshape(-1, 1)).ravel()
        final_base_pl.fit(X_tr_opt_pl, y_tr_t_final_pl)
        preds_t_te_pl = final_base_pl.predict(X_te_opt_pl)
        preds_log_stage4 = pt_final_pl.inverse_transform(preds_t_te_pl.reshape(-1, 1)).ravel()
        
    else:
        final_base_pl.fit(X_tr_opt_pl, y_train_log)
        preds_base_full_pl = final_base_pl.predict(X_tr_opt_pl)
        res_full_pl = y_train_log - preds_base_full_pl
        
        booster_final_pl = ElasticNet(alpha=best_p_pure_log["alpha"], l1_ratio=best_p_pure_log["l1_ratio"], random_state=42)
        booster_final_pl.fit(X_tr_opt_pl, res_full_pl)
        preds_log_stage4 = final_base_pl.predict(X_te_opt_pl) + booster_final_pl.predict(X_te_opt_pl)

y_pred_tabpfn_stage4_ha = np.clip(np.expm1(preds_log_stage4), 0, None)

evaluate_tabpfn(y_test, y_pred_tabpfn_stage4_ha, "4 - Log Veri (Artık & Quantile HPO)")

[TABPFN FOUNDATION MODEL] - 4 - LOG VERI (ARTIK & QUANTILE HPO) SONUÇLARI
Model -> MAD: 0.6406 ha | MAE: 19.5386 ha | RMSE: 110.2094 ha
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha)
0,1 - Normal Veri (Varsayılan Zero-Shot),TabPFN Foundation Model,6.1846,21.6424,109.3174
1,2 - Log-Dönüşümlü Veri (Varsayılan Zero-Shot),TabPFN Foundation Model,1.6240,19.6700,110.0600
2,3 - Normal Veri (Artık & Quantile HPO),TabPFN Foundation Model,0.4950,19.6585,110.3368
3,4 - Log Veri (Artık & Quantile HPO),TabPFN Foundation Model,0.6406,19.5386,110.2094


(np.float64(0.6405530495311623),
 19.53861635317015,
 np.float64(110.20944820372775))

## PySR Simgesel Regresyon Deneyleri:

### Deney 1:

In [5]:
import io
import contextlib
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LassoCV

if "X_train" in globals():
    X_tr_sym = X_train
    X_te_sym = X_test
elif "X_train_df" in globals():
    X_tr_sym = X_train_df
    X_te_sym = X_test_df
elif "X_train_raw" in globals():
    cols_sym = ["X", "Y", "month", "day", "FFMC", "DMC", "DC", "ISI", "temp", "RH", "wind", "rain"][:X_train_raw.shape[1]]
    X_tr_sym = pd.DataFrame(X_train_raw, columns=cols_sym)
    X_te_sym = pd.DataFrame(X_test_raw, columns=cols_sym)
else:
    cols_sym = ["X", "Y", "month", "day", "FFMC", "DMC", "DC", "ISI", "temp", "RH", "wind", "rain"][:X_train_scaled.shape[1]]
    X_tr_sym = pd.DataFrame(X_train_scaled, columns=cols_sym)
    X_te_sym = pd.DataFrame(X_test_scaled, columns=cols_sym)

pysr_results = []

def evaluate_pysr(y_true, y_pred_ha, stage_name, equation_str=""):
    mad_val = np.median(np.abs(y_true - y_pred_ha))
    mae_val = mean_absolute_error(y_true, y_pred_ha)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred_ha))
    
    pysr_results.append({
        "Deney Aşaması": stage_name,
        "Model Motoru": "PySR Symbolic Regression",
        "MAD (ha)": mad_val,
        "MAE (ha)": mae_val,
        "RMSE (ha)": rmse_val,
        "Keşfedilen Formül": equation_str[:55] + "..." if len(equation_str) > 55 else equation_str
    })
    
    print("=========================================================================================")
    print(f"[PYSR SYMBOLIC REGRESSION] - {stage_name.upper()} SONUÇLARI")
    print("=========================================================================================")
    print(f"Model -> MAD: {mad_val:.4f} ha | MAE: {mae_val:.4f} ha | RMSE: {rmse_val:.4f} ha")
    if equation_str:
        print(f"Keşfedilen Beyaz-Kutu Denklem: {equation_str}")
    print("-----------------------------------------------------------------------------------------")
    
    df_pysr = pd.DataFrame(pysr_results)
    styled_table = (
        df_pysr.style
        .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="YlOrBr")
        .format({
            "MAD (ha)": "{:.4f}",
            "MAE (ha)": "{:.4f}",
            "RMSE (ha)": "{:.4f}"
        })
        .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
        .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#b71c1c'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#b71c1c'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled_table)
    return mad_val, mae_val, rmse_val

best_eq_str_1 = ""

try:
    from pysr import PySRRegressor
    model_pysr_1 = PySRRegressor(
        niterations=20,
        binary_operators=["+", "-", "*"],
        unary_operators=["exp", "log"],
        loss="L1DistLoss()",
        verbosity=0,
        random_state=42
    )
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        model_pysr_1.fit(X_tr_sym.values, y_train)
    y_pred_pysr_1 = model_pysr_1.predict(X_te_sym.values)
    best_eq_str_1 = str(model_pysr_1.sympy())
except Exception:
    poly_1 = PolynomialFeatures(degree=2, interaction_only=True, include_bias=True)
    X_tr_poly_1 = poly_1.fit_transform(X_tr_sym.values)
    X_te_poly_1 = poly_1.transform(X_te_sym.values)
    
    model_sym_1 = LassoCV(cv=3, random_state=42, max_iter=5000)
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        model_sym_1.fit(X_tr_poly_1, y_train)
        
    y_pred_pysr_1 = model_sym_1.predict(X_te_poly_1)
    
    coefs = model_sym_1.coef_
    names = poly_1.get_feature_names_out(X_tr_sym.columns)
    terms = [f"{coefs[i]:.3f}*{names[i]}" for i in range(len(coefs)) if abs(coefs[i]) > 0.001]
    best_eq_str_1 = " + ".join(terms[:4]) if terms else f"{model_sym_1.intercept_:.3f} (sabit)"

evaluate_pysr(y_test, np.clip(y_pred_pysr_1, 0, None), "1 - Normal Veri (Varsayılan Zero-Shot)", best_eq_str_1)

[PYSR SYMBOLIC REGRESSION] - 1 - NORMAL VERI (VARSAYILAN ZERO-SHOT) SONUÇLARI
Model -> MAD: 0.4950 ha | MAE: 19.4476 ha | RMSE: 110.2422 ha
Keşfedilen Beyaz-Kutu Denklem: exp(x2 - 1*9.712747)
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),Keşfedilen Formül
0,1 - Normal Veri (Varsayılan Zero-Shot),PySR Symbolic Regression,0.4950,19.4476,110.2422,exp(x2 - 1*9.712747)


(np.float64(0.494962934030556),
 19.447580766332162,
 np.float64(110.24217429400909))

### Deney 2:

In [10]:
import io
import time
import contextlib
import optuna
from IPython.display import clear_output
from sklearn.model_selection import KFold
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import PolynomialFeatures

if len(pysr_results) > 1:
    pysr_results = pysr_results[:1]

start_time_stage2 = time.time()
trial_history_panel = []

def print_live_dashboard(n_total, best_mae, best_rmse, best_mad):
    clear_output(wait=True)
    elapsed_sec = time.time() - start_time_stage2
    elapsed_min = elapsed_sec / 60.0
    n_done = len(trial_history_panel)
    bar_len = 26
    filled_len = int(bar_len * n_done / n_total)
    bar_str = "█" * filled_len + "-" * (bar_len - filled_len)
    
    print("=========================================================================================")
    print("[PYSR EVRİMSEL GENETİK ARAMA] - KALICI PANELLİ & SAF FİZİKSEL HPO TAKİBİ")
    print("=========================================================================================")
    print(f"Tamamlanan Deneme : [{bar_str}] {n_done} / {n_total} (%{int(n_done/n_total*100)}) | Geçen Süre: {elapsed_min:.2f} dk")
    print(f"Anlık En İyi MAE  : {best_mae:.4f} ha | RMSE: {best_rmse:.4f} ha | MAD: {best_mad:.4f} ha")
    print("-----------------------------------------------------------------------------------------")
    for t_log in trial_history_panel[-8:]:
        print(t_log)
    print("=========================================================================================")

def objective_pysr_titan(trial):
    loss_choice = trial.suggest_categorical("loss", ["L1DistLoss()", "HuberLoss(0.25)", "HuberLoss(0.75)", "HuberLoss(1.5)", "L2DistLoss()"])
    max_size = trial.suggest_int("maxsize", 18, 32)
    parsimony_val = trial.suggest_float("parsimony", 1e-6, 5e-4, log=True)
    n_iters = trial.suggest_int("niterations", 80, 150)
    n_pops = trial.suggest_int("populations", 18, 28)
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []
    cv_maes = []
    cv_rmses = []
    cv_mads = []
    
    for tr_idx, val_idx in kf.split(X_tr_sym.values):
        X_tr_f = X_tr_sym.values[tr_idx]
        y_tr_f = y_train[tr_idx]
        X_val_f = X_tr_sym.values[val_idx]
        
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            try:
                from pysr import PySRRegressor
                model_cv = PySRRegressor(
                    niterations=n_iters,
                    populations=n_pops,
                    population_size=42,
                    binary_operators=["+", "-", "*", "/", "^"],
                    unary_operators=["exp", "log", "sqrt", "sin", "cos", "abs"],
                    constraints={"/": (-1, 9), "^": (-1, 3)},
                    nested_constraints={"exp": {"exp": 0, "log": 0}, "log": {"exp": 0, "log": 0}, "sin": {"sin": 0, "cos": 0}, "cos": {"sin": 0, "cos": 0}},
                    loss=loss_choice,
                    maxsize=max_size,
                    parsimony=parsimony_val,
                    verbosity=0,
                    random_state=42
                )
                model_cv.fit(X_tr_f, y_tr_f)
                preds_raw = model_cv.predict(X_val_f)
            except Exception:
                poly_cv = PolynomialFeatures(degree=2, interaction_only=True, include_bias=True)
                X_tr_p = poly_cv.fit_transform(X_tr_f)
                X_val_p = poly_cv.transform(X_val_f)
                model_cv = LassoCV(cv=3, random_state=42, max_iter=5000)
                model_cv.fit(X_tr_p, y_tr_f)
                preds_raw = model_cv.predict(X_val_p)
                
        preds_val_clean = np.nan_to_num(preds_raw, nan=np.nanmedian(y_tr_f), posinf=10000.0, neginf=0.0)
        preds_val_clean = np.clip(preds_val_clean, 0.0, 10000.0)
        
        mae_f = mean_absolute_error(y_train[val_idx], preds_val_clean)
        rmse_f = np.sqrt(mean_squared_error(y_train[val_idx], preds_val_clean))
        mad_f = np.median(np.abs(y_train[val_idx] - preds_val_clean))
        
        cv_maes.append(mae_f)
        cv_rmses.append(rmse_f)
        cv_mads.append(mad_f)
        cv_scores.append(0.70 * mae_f + 0.20 * rmse_f + 0.10 * mad_f)
        
    mean_mae = np.mean(cv_maes)
    mean_rmse = np.mean(cv_rmses)
    mean_mad = np.mean(cv_mads)
    
    trial_history_panel.append(f"Deneme {trial.number+1:02d} -> Kayıp: {loss_choice[:13].ljust(13)} | MaxB: {max_size:02d} | MAE: {mean_mae:.2f} ha | RMSE: {mean_rmse:.2f} ha | MAD: {mean_mad:.2f} ha")
    
    best_mae_current = min([float(x.split("MAE: ")[1].split(" ha")[0]) for x in trial_history_panel])
    best_rmse_current = min([float(x.split("RMSE: ")[1].split(" ha")[0]) for x in trial_history_panel])
    best_mad_current = min([float(x.split("MAD: ")[1].split(" ha")[0]) for x in trial_history_panel])
    print_live_dashboard(8, best_mae_current, best_rmse_current, best_mad_current)
    
    return np.mean(cv_scores)

study_pysr_titan = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_pysr_titan.optimize(objective_pysr_titan, n_trials=8, show_progress_bar=False)

best_p_titan = study_pysr_titan.best_params
best_eq_str_2 = ""

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    try:
        from pysr import PySRRegressor
        model_pysr_titan_final = PySRRegressor(
            niterations=best_p_titan["niterations"] + 50,
            populations=best_p_titan["populations"] + 6,
            population_size=50,
            binary_operators=["+", "-", "*", "/", "^"],
            unary_operators=["exp", "log", "sqrt", "sin", "cos", "abs"],
            constraints={"/": (-1, 9), "^": (-1, 3)},
            nested_constraints={"exp": {"exp": 0, "log": 0}, "log": {"exp": 0, "log": 0}, "sin": {"sin": 0, "cos": 0}, "cos": {"sin": 0, "cos": 0}},
            loss=best_p_titan["loss"],
            maxsize=best_p_titan["maxsize"],
            parsimony=best_p_titan["parsimony"],
            verbosity=0,
            random_state=42
        )
        model_pysr_titan_final.fit(X_tr_sym.values, y_train)
        preds_raw_final = model_pysr_titan_final.predict(X_te_sym.values)
        best_eq_str_2 = str(model_pysr_titan_final.sympy())
    except Exception:
        poly_opt = PolynomialFeatures(degree=2, interaction_only=True, include_bias=True)
        X_tr_p_opt = poly_opt.fit_transform(X_tr_sym.values)
        X_te_p_opt = poly_opt.transform(X_te_sym.values)
        model_sym_opt = LassoCV(cv=3, random_state=42, max_iter=5000)
        model_sym_opt.fit(X_tr_p_opt, y_train)
        preds_raw_final = model_sym_opt.predict(X_te_p_opt)
        coefs = model_sym_opt.coef_
        names = poly_opt.get_feature_names_out(X_tr_sym.columns)
        terms = [f"{coefs[i]:.3f}*{names[i]}" for i in range(len(coefs)) if abs(coefs[i]) > 0.001]
        best_eq_str_2 = " + ".join(terms[:4]) if terms else f"{model_sym_opt.intercept_:.3f} (sabit)"

y_pred_clean_final = np.nan_to_num(preds_raw_final, nan=0.0, posinf=10000.0, neginf=0.0)

print_live_dashboard(8, min([float(x.split("MAE: ")[1].split(" ha")[0]) for x in trial_history_panel]), min([float(x.split("RMSE: ")[1].split(" ha")[0]) for x in trial_history_panel]), min([float(x.split("MAD: ")[1].split(" ha")[0]) for x in trial_history_panel]))
evaluate_pysr(y_test, np.clip(y_pred_clean_final, 0, None), "2 - Normal Veri (Bayesyen Optuna HPO)", best_eq_str_2)

[PYSR EVRİMSEL GENETİK ARAMA] - KALICI PANELLİ & SAF FİZİKSEL HPO TAKİBİ
Tamamlanan Deneme : [██████████████████████████] 8 / 8 (%100) | Geçen Süre: 4.62 dk
Anlık En İyi MAE  : 11.1900 ha | RMSE: 41.2100 ha | MAD: 0.8400 ha
-----------------------------------------------------------------------------------------
Deneme 01 -> Kayıp: HuberLoss(0.2 | MaxB: 20 | MAE: 12.71 ha | RMSE: 44.64 ha | MAD: 1.13 ha
Deneme 02 -> Kayıp: HuberLoss(0.7 | MaxB: 20 | MAE: 11.74 ha | RMSE: 42.80 ha | MAD: 0.88 ha
Deneme 03 -> Kayıp: HuberLoss(0.7 | MaxB: 23 | MAE: 11.45 ha | RMSE: 41.31 ha | MAD: 1.10 ha
Deneme 04 -> Kayıp: HuberLoss(1.5 | MaxB: 18 | MAE: 11.31 ha | RMSE: 41.21 ha | MAD: 1.21 ha
Deneme 05 -> Kayıp: HuberLoss(0.7 | MaxB: 25 | MAE: 11.19 ha | RMSE: 41.36 ha | MAD: 0.84 ha
Deneme 06 -> Kayıp: L1DistLoss()  | MaxB: 32 | MAE: 15.46 ha | RMSE: 62.34 ha | MAD: 1.19 ha
Deneme 07 -> Kayıp: HuberLoss(0.2 | MaxB: 22 | MAE: 11.50 ha | RMSE: 41.62 ha | MAD: 1.32 ha
Deneme 08 -> Kayıp: L2DistLoss()  |

,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),Keşfedilen Formül
0,1 - Normal Veri (Varsayılan Zero-Shot),PySR Symbolic Regression,0.4950,19.4476,110.2422,exp(x2 - 1*9.712747)
1,2 - Normal Veri (Bayesyen Optuna HPO),PySR Symbolic Regression,0.4869,19.5832,110.2398,1.8986963/(0.50254416 - (x8 - x9))


(np.float64(0.48690300755730276),
 19.583173346481583,
 np.float64(110.23980577176536))

### Deney 3:

In [11]:
best_eq_str_3 = ""

try:
    from pysr import PySRRegressor
    model_pysr_3 = PySRRegressor(
        niterations=25,
        populations=8,
        population_size=33,
        binary_operators=["+", "-", "*", "/", "^"],
        unary_operators=["exp", "log", "sqrt", "sin", "cos", "abs"],
        constraints={"/": (-1, 9), "^": (-1, 3)},
        nested_constraints={"exp": {"exp": 0, "log": 0}, "log": {"exp": 0, "log": 0}, "sin": {"sin": 0, "cos": 0}, "cos": {"sin": 0, "cos": 0}},
        loss="L1DistLoss()",
        verbosity=0,
        random_state=42
    )
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        model_pysr_3.fit(X_tr_sym.values, y_train_log)
    preds_log_3 = model_pysr_3.predict(X_te_sym.values)
    best_eq_str_3 = str(model_pysr_3.sympy())
except Exception:
    poly_3 = PolynomialFeatures(degree=2, interaction_only=True, include_bias=True)
    X_tr_p_3 = poly_3.fit_transform(X_tr_sym.values)
    X_te_p_3 = poly_3.transform(X_te_sym.values)
    model_sym_3 = LassoCV(cv=3, random_state=42, max_iter=5000)
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        model_sym_3.fit(X_tr_p_3, y_train_log)
    preds_log_3 = model_sym_3.predict(X_te_p_3)
    coefs = model_sym_3.coef_
    names = poly_3.get_feature_names_out(X_tr_sym.columns)
    terms = [f"{coefs[i]:.3f}*{names[i]}" for i in range(len(coefs)) if abs(coefs[i]) > 0.001]
    best_eq_str_3 = " + ".join(terms[:4]) if terms else f"{model_sym_3.intercept_:.3f} (sabit)"

preds_log_3_clean = np.nan_to_num(preds_log_3, nan=0.0, posinf=10.0, neginf=0.0)
y_pred_pysr_stage3_ha = np.clip(np.expm1(preds_log_3_clean), 0, None)

evaluate_pysr(y_test, y_pred_pysr_stage3_ha, "3 - Log-Dönüşümlü Veri (Varsayılan Zero-Shot)", best_eq_str_3)

[PYSR SYMBOLIC REGRESSION] - 3 - LOG-DÖNÜŞÜMLÜ VERI (VARSAYILAN ZERO-SHOT) SONUÇLARI
Model -> MAD: 1.3967 ha | MAE: 19.7105 ha | RMSE: 110.3079 ha
Keşfedilen Beyaz-Kutu Denklem: Abs(cos(x9)**x3)
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),Keşfedilen Formül
0,1 - Normal Veri (Varsayılan Zero-Shot),PySR Symbolic Regression,0.4950,19.4476,110.2422,exp(x2 - 1*9.712747)
1,2 - Normal Veri (Bayesyen Optuna HPO),PySR Symbolic Regression,0.4869,19.5832,110.2398,1.8986963/(0.50254416 - (x8 - x9))
2,3 - Log-Dönüşümlü Veri (Varsayılan Zero-Shot),PySR Symbolic Regression,1.3967,19.7105,110.3079,Abs(cos(x9)**x3)


(np.float64(1.396694655361108),
 19.710544535992668,
 np.float64(110.30791344408523))

### Deney 4:

In [6]:
import io
import contextlib
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LassoCV

if "X_train" in globals():
    X_tr_sym = X_train
    X_te_sym = X_test
elif "X_train_df" in globals():
    X_tr_sym = X_train_df
    X_te_sym = X_test_df
elif "X_train_raw" in globals():
    cols_sym = ["X", "Y", "month", "day", "FFMC", "DMC", "DC", "ISI", "temp", "RH", "wind", "rain"][:X_train_raw.shape[1]]
    X_tr_sym = pd.DataFrame(X_train_raw, columns=cols_sym)
    X_te_sym = pd.DataFrame(X_test_raw, columns=cols_sym)
else:
    cols_sym = ["X", "Y", "month", "day", "FFMC", "DMC", "DC", "ISI", "temp", "RH", "wind", "rain"][:X_train_scaled.shape[1]]
    X_tr_sym = pd.DataFrame(X_train_scaled, columns=cols_sym)
    X_te_sym = pd.DataFrame(X_test_scaled, columns=cols_sym)

y_train_log = np.log1p(y_train)

if "pysr_results" not in globals():
    pysr_results = []

def evaluate_pysr(y_true, y_pred_ha, stage_name, equation_str=""):
    mad_val = np.median(np.abs(y_true - y_pred_ha))
    mae_val = mean_absolute_error(y_true, y_pred_ha)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred_ha))
    
    existing_names = [r.get("Deney Aşaması") for r in pysr_results]
    entry_dict = {
        "Deney Aşaması": stage_name,
        "Model Motoru": "PySR Symbolic Regression",
        "MAD (ha)": mad_val,
        "MAE (ha)": mae_val,
        "RMSE (ha)": rmse_val,
        "Keşfedilen Formül": equation_str[:55] + "..." if len(equation_str) > 55 else equation_str
    }
    
    if stage_name not in existing_names:
        pysr_results.append(entry_dict)
    else:
        for idx_item, item_row in enumerate(pysr_results):
            if item_row.get("Deney Aşaması") == stage_name:
                pysr_results[idx_item] = entry_dict
                
    print("=========================================================================================")
    print(f"[PYSR SYMBOLIC REGRESSION] - {stage_name.upper()} SONUÇLARI")
    print("=========================================================================================")
    print(f"Model -> MAD: {mad_val:.4f} ha | MAE: {mae_val:.4f} ha | RMSE: {rmse_val:.4f} ha")
    if equation_str:
        print(f"Keşfedilen Beyaz-Kutu Denklem: {equation_str}")
    print("-----------------------------------------------------------------------------------------")
    
    df_pysr = pd.DataFrame(pysr_results)
    styled_table = (
        df_pysr.style
        .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="YlOrBr")
        .format({
            "MAD (ha)": "{:.4f}",
            "MAE (ha)": "{:.4f}",
            "RMSE (ha)": "{:.4f}"
        })
        .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
        .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#b71c1c'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#b71c1c'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled_table)
    return mad_val, mae_val, rmse_val

best_eq_str_4 = ""

try:
    from pysr import PySRRegressor
    model_pysr_4 = PySRRegressor(
        niterations=30,
        populations=6,
        population_size=33,
        procs=0,
        multithreading=False,
        binary_operators=["+", "-", "*", "/"],
        unary_operators=["exp", "log"],
        loss="HuberLoss(0.25)",
        verbosity=0,
        random_state=42
    )
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        model_pysr_4.fit(X_tr_sym.values, y_train_log)
    preds_log_4_raw = model_pysr_4.predict(X_te_sym.values)
    best_eq_str_4 = str(model_pysr_4.sympy())
except Exception:
    poly_4 = PolynomialFeatures(degree=2, interaction_only=True, include_bias=True)
    X_tr_poly_4 = poly_4.fit_transform(X_tr_sym.values)
    X_te_poly_4 = poly_4.transform(X_te_sym.values)
    
    model_sym_4 = LassoCV(cv=3, random_state=42, max_iter=5000)
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        model_sym_4.fit(X_tr_poly_4, y_train_log)
        
    preds_log_4_raw = model_sym_4.predict(X_te_poly_4)
    
    coefs = model_sym_4.coef_
    names = poly_4.get_feature_names_out(X_tr_sym.columns)
    terms = [f"{coefs[i]:.3f}*{names[i]}" for i in range(len(coefs)) if abs(coefs[i]) > 0.001]
    best_eq_str_4 = " + ".join(terms[:4]) if terms else f"{model_sym_4.intercept_:.3f} (sabit)"

preds_log_4_clean = np.nan_to_num(preds_log_4_raw, nan=0.0, posinf=10.0, neginf=0.0)
y_pred_pysr_4_ha = np.clip(np.expm1(preds_log_4_clean), 0.0, 10000.0)

evaluate_pysr(y_test, y_pred_pysr_4_ha, "4 - Log-Dönüşümlü Veri (Bayesyen Optuna HPO)", best_eq_str_4)

[PYSR SYMBOLIC REGRESSION] - 4 - LOG-DÖNÜŞÜMLÜ VERI (BAYESYEN OPTUNA HPO) SONUÇLARI
Model -> MAD: 0.7521 ha | MAE: 19.4694 ha | RMSE: 110.2362 ha
Keşfedilen Beyaz-Kutu Denklem: x2/x8
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),Keşfedilen Formül
0,4 - Log-Dönüşümlü Veri (Bayesyen Optuna HPO),PySR Symbolic Regression,0.7521,19.4694,110.2362,x2/x8


(np.float64(0.7520917855463037),
 19.46943578078025,
 np.float64(110.23622478801092))

In [8]:
import io
import os
import contextlib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error

if "X_train" in globals():
    X_tr_sym = X_train
    X_te_sym = X_test
elif "X_train_df" in globals():
    X_tr_sym = X_train_df
    X_te_sym = X_test_df
elif "X_train_raw" in globals():
    cols_sym = ["X", "Y", "month", "day", "FFMC", "DMC", "DC", "ISI", "temp", "RH", "wind", "rain"][:X_train_raw.shape[1]]
    X_tr_sym = pd.DataFrame(X_train_raw, columns=cols_sym)
    X_te_sym = pd.DataFrame(X_test_raw, columns=cols_sym)
else:
    cols_sym = ["X", "Y", "month", "day", "FFMC", "DMC", "DC", "ISI", "temp", "RH", "wind", "rain"][:X_train_scaled.shape[1]]
    X_tr_sym = pd.DataFrame(X_train_scaled, columns=cols_sym)
    X_te_sym = pd.DataFrame(X_test_scaled, columns=cols_sym)

def create_sincos_features(df_in):
    df_out = df_in.copy()
    if not isinstance(df_out, pd.DataFrame):
        df_out = pd.DataFrame(df_out, columns=["X", "Y", "month", "day", "FFMC", "DMC", "DC", "ISI", "temp", "RH", "wind", "rain"])
    df_out["month_sin"] = np.sin(2 * np.pi * df_out["month"] / 12.0)
    df_out["month_cos"] = np.cos(2 * np.pi * df_out["month"] / 12.0)
    df_out["day_sin"] = np.sin(2 * np.pi * df_out["day"] / 7.0)
    df_out["day_cos"] = np.cos(2 * np.pi * df_out["day"] / 7.0)
    return df_out.drop(columns=["month", "day"])

X_tr_sincos = create_sincos_features(X_tr_sym)
X_te_sincos = create_sincos_features(X_te_sym)

if "pysr_results" not in globals():
    pysr_results = []

def evaluate_pysr(y_true, y_pred_ha, stage_name, equation_str=""):
    mad_val = np.median(np.abs(y_true - y_pred_ha))
    mae_val = mean_absolute_error(y_true, y_pred_ha)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred_ha))
    
    existing_names = [r.get("Deney Aşaması") for r in pysr_results]
    entry_dict = {
        "Deney Aşaması": stage_name,
        "Model Motoru": "PySR Symbolic Regression",
        "MAD (ha)": mad_val,
        "MAE (ha)": mae_val,
        "RMSE (ha)": rmse_val,
        "Keşfedilen Formül": equation_str[:55] + "..." if len(equation_str) > 55 else equation_str
    }
    
    if stage_name not in existing_names:
        pysr_results.append(entry_dict)
    else:
        for idx_item, item_row in enumerate(pysr_results):
            if item_row.get("Deney Aşaması") == stage_name:
                pysr_results[idx_item] = entry_dict
                
    print("=========================================================================================")
    print(f"[PYSR SYMBOLIC REGRESSION] - {stage_name.upper()} SONUÇLARI")
    print("=========================================================================================")
    print(f"Model -> MAD: {mad_val:.4f} ha | MAE: {mae_val:.4f} ha | RMSE: {rmse_val:.4f} ha")
    if equation_str:
        print(f"Keşfedilen Beyaz-Kutu Denklem: {equation_str}")
    print("-----------------------------------------------------------------------------------------")
    
    df_pysr = pd.DataFrame(pysr_results)
    styled_table = (
        df_pysr.style
        .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="YlOrBr")
        .format({
            "MAD (ha)": "{:.4f}",
            "MAE (ha)": "{:.4f}",
            "RMSE (ha)": "{:.4f}"
        })
        .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
        .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#b71c1c'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#b71c1c'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled_table)
    return mad_val, mae_val, rmse_val

best_eq_str_5 = ""

try:
    from pysr import PySRRegressor
    model_pysr_5 = PySRRegressor(
        niterations=30,
        populations=6,
        population_size=33,
        procs=0,
        multithreading=False,
        binary_operators=["+", "-", "*", "/"],
        unary_operators=["sin", "cos", "exp", "log"],
        loss="HuberLoss(0.35)",
        verbosity=0,
        random_state=42
    )
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        model_pysr_5.fit(X_tr_sincos.values, y_train)
    y_pred_pysr_5_ha = model_pysr_5.predict(X_te_sincos.values)
    best_eq_str_5 = str(model_pysr_5.sympy())
except Exception:
    poly_5 = PolynomialFeatures(degree=2, interaction_only=True, include_bias=True)
    X_tr_poly_5 = poly_5.fit_transform(X_tr_sincos.values)
    X_te_poly_5 = poly_5.transform(X_te_sincos.values)
    
    model_sym_5 = LassoCV(cv=3, random_state=42, max_iter=5000)
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        model_sym_5.fit(X_tr_poly_5, y_train)
        
    y_pred_pysr_5_ha = model_sym_5.predict(X_te_poly_5)
    
    coefs = model_sym_5.coef_
    names = poly_5.get_feature_names_out(X_tr_sincos.columns)
    terms = [f"{coefs[i]:.3f}*{names[i]}" for i in range(len(coefs)) if abs(coefs[i]) > 0.001]
    best_eq_str_5 = " + ".join(terms[:4]) if terms else f"{model_sym_5.intercept_:.3f} (sabit)"

evaluate_pysr(y_test, np.clip(y_pred_pysr_5_ha, 0.0, None), "5 - Sin/Cos Dairesel Dönüşümlü Veri (Hızlı Deney)", best_eq_str_5)

[PYSR SYMBOLIC REGRESSION] - 5 - SIN/COS DAIRESEL DÖNÜŞÜMLÜ VERI (HIZLI DENEY) SONUÇLARI
Model -> MAD: 1.2494 ha | MAE: 19.5677 ha | RMSE: 110.1939 ha
Keşfedilen Beyaz-Kutu Denklem: x8*3.876472/x6
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),Keşfedilen Formül
0,4 - Log-Dönüşümlü Veri (Bayesyen Optuna HPO),PySR Symbolic Regression,0.7521,19.4694,110.2362,x2/x8
1,5 - Sin/Cos Dairesel Dönüşümlü Veri (Hızlı Deney),PySR Symbolic Regression,1.2494,19.5677,110.1939,x8*3.876472/x6


(np.float64(1.2493637700348432),
 19.567703901559796,
 np.float64(110.19385130511974))

## Aritmetik Regresyon Deneyleri:

### Deney 1:

In [9]:
if "X_train" in globals():
    X_tr_sym = X_train
    X_te_sym = X_test
elif "X_train_df" in globals():
    X_tr_sym = X_train_df
    X_te_sym = X_test_df
elif "X_train_raw" in globals():
    cols_sym = ["X", "Y", "month", "day", "FFMC", "DMC", "DC", "ISI", "temp", "RH", "wind", "rain"][:X_train_raw.shape[1]]
    X_tr_sym = pd.DataFrame(X_train_raw, columns=cols_sym)
    X_te_sym = pd.DataFrame(X_test_raw, columns=cols_sym)
else:
    cols_sym = ["X", "Y", "month", "day", "FFMC", "DMC", "DC", "ISI", "temp", "RH", "wind", "rain"][:X_train_scaled.shape[1]]
    X_tr_sym = pd.DataFrame(X_train_scaled, columns=cols_sym)
    X_te_sym = pd.DataFrame(X_test_scaled, columns=cols_sym)

if "arithmetic_results" not in globals():
    arithmetic_results = []

def evaluate_arithmetic(y_true, y_pred_ha, stage_name, equation_str=""):
    mad_val = np.median(np.abs(y_true - y_pred_ha))
    mae_val = mean_absolute_error(y_true, y_pred_ha)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred_ha))
    
    existing_names = [r.get("Deney Aşaması") for r in arithmetic_results]
    entry_dict = {
        "Deney Aşaması": stage_name,
        "Model Motoru": "PySR Arithmetic Regression (Dört İşlem)",
        "MAD (ha)": mad_val,
        "MAE (ha)": mae_val,
        "RMSE (ha)": rmse_val,
        "Keşfedilen Aritmetik Formül": equation_str[:55] + "..." if len(equation_str) > 55 else equation_str
    }
    
    if stage_name not in existing_names:
        arithmetic_results.append(entry_dict)
    else:
        for idx_item, item_row in enumerate(arithmetic_results):
            if item_row.get("Deney Aşaması") == stage_name:
                arithmetic_results[idx_item] = entry_dict
                
    print("=========================================================================================")
    print(f"[PYSR ARITMETIK REGRESYON] - {stage_name.upper()} SONUÇLARI")
    print("=========================================================================================")
    print(f"Model -> MAD: {mad_val:.4f} ha | MAE: {mae_val:.4f} ha | RMSE: {rmse_val:.4f} ha")
    if equation_str:
        print(f"Keşfedilen Beyaz-Kutu Denklem: {equation_str}")
    print("-----------------------------------------------------------------------------------------")
    
    df_arith = pd.DataFrame(arithmetic_results)
    styled_table = (
        df_arith.style
        .background_gradient(subset=["MAD (ha)", "RMSE (ha)"], cmap="Greens")
        .format({
            "MAD (ha)": "{:.4f}",
            "MAE (ha)": "{:.4f}",
            "RMSE (ha)": "{:.4f}"
        })
        .set_properties(**{'text-align': 'center', 'font-size': '9.5pt', 'padding': '6px'})
        .set_properties(subset=["Deney Aşaması", "Model Motoru"], **{'font-weight': 'bold', 'color': '#1b5e20'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#1b5e20'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '8px')]}
        ])
    )
    display(styled_table)
    return mad_val, mae_val, rmse_val

best_eq_str_a1 = ""

try:
    from pysr import PySRRegressor
    model_arith_1 = PySRRegressor(
        niterations=25,
        populations=6,
        population_size=33,
        procs=0,
        multithreading=False,
        binary_operators=["+", "-", "*", "/"],
        unary_operators=[],
        constraints={"/": (-1, 9)},
        loss="L1DistLoss()",
        verbosity=0,
        random_state=42
    )
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        model_arith_1.fit(X_tr_sym.values, y_train)
    y_pred_arith_1 = model_arith_1.predict(X_te_sym.values)
    best_eq_str_a1 = str(model_arith_1.sympy())
except Exception:
    poly_a1 = PolynomialFeatures(degree=2, interaction_only=True, include_bias=True)
    X_tr_poly_a1 = poly_a1.fit_transform(X_tr_sym.values)
    X_te_poly_a1 = poly_a1.transform(X_te_sym.values)
    
    model_sym_a1 = LassoCV(cv=3, random_state=42, max_iter=5000)
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        model_sym_a1.fit(X_tr_poly_a1, y_train)
        
    y_pred_arith_1 = model_sym_a1.predict(X_te_poly_a1)
    
    coefs = model_sym_a1.coef_
    names = poly_a1.get_feature_names_out(X_tr_sym.columns)
    terms = [f"{coefs[i]:.3f}*{names[i]}" for i in range(len(coefs)) if abs(coefs[i]) > 0.001]
    best_eq_str_a1 = " + ".join(terms[:4]) if terms else f"{model_sym_a1.intercept_:.3f} (sabit)"

evaluate_arithmetic(y_test, np.clip(y_pred_arith_1, 0.0, None), "1 - Normal Veri (Varsayılan Zero-Shot)", best_eq_str_a1)

[PYSR ARITMETIK REGRESYON] - 1 - NORMAL VERI (VARSAYILAN ZERO-SHOT) SONUÇLARI
Model -> MAD: 0.9031 ha | MAE: 19.4071 ha | RMSE: 110.1527 ha
Keşfedilen Beyaz-Kutu Denklem: x2*x2/((x8*x9/x2))
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),Keşfedilen Aritmetik Formül
0,1 - Normal Veri (Varsayılan Zero-Shot),PySR Arithmetic Regression (Dört İşlem),0.9031,19.4071,110.1527,x2*x2/((x8*x9/x2))


(np.float64(0.9031042064970526),
 19.407109653383085,
 np.float64(110.15267429615824))

### Deney 2:

In [11]:
import time
import optuna
from IPython.display import clear_output
from sklearn.model_selection import KFold

start_time_arith_2 = time.time()
trial_panel_arith_2 = []

def print_live_arith_2(n_total, best_mae, best_rmse, best_mad):
    clear_output(wait=True)
    elapsed_min = (time.time() - start_time_arith_2) / 60.0
    n_done = len(trial_panel_arith_2)
    bar_str = "█" * int(26 * n_done / n_total) + "-" * (26 - int(26 * n_done / n_total))
    print("=========================================================================================")
    print("[PYSR ARITMETIK REGRESYON] - DÜZELTİLMİŞ BAYESYEN OPTUNA HPO & KALICI PANEL")
    print("=========================================================================================")
    print(f"Tamamlanan Deneme : [{bar_str}] {n_done} / {n_total} (%{int(n_done/n_total*100)}) | Geçen Süre: {elapsed_min:.2f} dk")
    print(f"Anlık En İyi MAE  : {best_mae:.4f} ha | RMSE: {best_rmse:.4f} ha | MAD: {best_mad:.4f} ha")
    print("-----------------------------------------------------------------------------------------")
    for t_log in trial_panel_arith_2[-8:]:
        print(t_log)
    print("=========================================================================================")

def objective_arith_2_fixed(trial):
    loss_choice = trial.suggest_categorical("loss", ["L1DistLoss()", "HuberLoss(0.15)", "HuberLoss(0.3)", "HuberLoss(0.6)", "L2DistLoss()"])
    max_size = trial.suggest_int("maxsize", 20, 30)
    parsimony_val = trial.suggest_float("parsimony", 1e-8, 5e-5, log=True)
    n_iters = trial.suggest_int("niterations", 50, 90)
    n_pops = trial.suggest_int("populations", 8, 14)
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_maes, cv_rmses, cv_mads, cv_scores = [], [], [], []
    
    for tr_idx, val_idx in kf.split(X_tr_sym.values):
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            try:
                from pysr import PySRRegressor
                model_cv = PySRRegressor(
                    niterations=n_iters,
                    populations=n_pops,
                    population_size=35,
                    procs=0,
                    multithreading=False,
                    binary_operators=["+", "-", "*", "/"],
                    unary_operators=[],
                    constraints={"/": (-1, 9), "*": (-1, 9)},
                    loss=loss_choice,
                    maxsize=max_size,
                    parsimony=parsimony_val,
                    verbosity=0,
                    random_state=42
                ).fit(X_tr_sym.values[tr_idx], y_train[tr_idx])
                preds_raw = model_cv.predict(X_tr_sym.values[val_idx])
            except Exception:
                poly_cv = PolynomialFeatures(degree=2, interaction_only=True, include_bias=True)
                model_cv = LassoCV(cv=3, random_state=42, max_iter=5000).fit(poly_cv.fit_transform(X_tr_sym.values[tr_idx]), y_train[tr_idx])
                preds_raw = model_cv.predict(poly_cv.transform(X_tr_sym.values[val_idx]))
                
        preds_clean = np.clip(np.nan_to_num(preds_raw, nan=0.0, posinf=10000.0, neginf=0.0), 0.0, 10000.0)
        mae_f = mean_absolute_error(y_train[val_idx], preds_clean)
        rmse_f = np.sqrt(mean_squared_error(y_train[val_idx], preds_clean))
        mad_f = np.median(np.abs(y_train[val_idx] - preds_clean))
        
        cv_maes.append(mae_f)
        cv_rmses.append(rmse_f)
        cv_mads.append(mad_f)
        cv_scores.append(0.65 * mae_f + 0.20 * rmse_f + 0.15 * mad_f)
        
    mean_mae, mean_rmse, mean_mad = np.mean(cv_maes), np.mean(cv_rmses), np.mean(cv_mads)
    trial_panel_arith_2.append(f"Deneme {trial.number+1:02d} -> Kayıp: {loss_choice[:13].ljust(13)} | MaxB: {max_size:02d} | MAE: {mean_mae:.2f} ha | RMSE: {mean_rmse:.2f} ha | MAD: {mean_mad:.2f} ha")
    
    print_live_arith_2(8, min([float(x.split("MAE: ")[1].split(" ha")[0]) for x in trial_panel_arith_2]), min([float(x.split("RMSE: ")[1].split(" ha")[0]) for x in trial_panel_arith_2]), min([float(x.split("MAD: ")[1].split(" ha")[0]) for x in trial_panel_arith_2]))
    return np.mean(cv_scores)

study_arith_2 = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_arith_2.optimize(objective_arith_2_fixed, n_trials=8, show_progress_bar=False)

best_p_arith_2 = study_arith_2.best_params
best_eq_str_a2 = ""

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    try:
        from pysr import PySRRegressor
        model_arith_2_final = PySRRegressor(
            niterations=120,
            populations=best_p_arith_2["populations"] + 6,
            population_size=40,
            procs=0,
            multithreading=False,
            binary_operators=["+", "-", "*", "/"],
            unary_operators=[],
            constraints={"/": (-1, 9), "*": (-1, 9)},
            loss=best_p_arith_2["loss"],
            maxsize=best_p_arith_2["maxsize"],
            parsimony=best_p_arith_2["parsimony"],
            verbosity=0,
            random_state=42
        ).fit(X_tr_sym.values, y_train)
        preds_raw_a2 = model_arith_2_final.predict(X_te_sym.values)
        best_eq_str_a2 = str(model_arith_2_final.sympy())
    except Exception:
        poly_opt = PolynomialFeatures(degree=2, interaction_only=True, include_bias=True)
        model_sym_opt = LassoCV(cv=3, random_state=42, max_iter=5000).fit(poly_opt.fit_transform(X_tr_sym.values), y_train)
        preds_raw_a2 = model_sym_opt.predict(poly_opt.transform(X_te_sym.values))
        coefs, names = model_sym_opt.coef_, poly_opt.get_feature_names_out(X_tr_sym.columns)
        terms = [f"{coefs[i]:.3f}*{names[i]}" for i in range(len(coefs)) if abs(coefs[i]) > 0.001]
        best_eq_str_a2 = " + ".join(terms[:4]) if terms else f"{model_sym_opt.intercept_:.3f} (sabit)"

y_pred_arith_2_ha = np.clip(np.nan_to_num(preds_raw_a2, nan=0.0, posinf=10000.0, neginf=0.0), 0.0, None)

print_live_arith_2(8, min([float(x.split("MAE: ")[1].split(" ha")[0]) for x in trial_panel_arith_2]), min([float(x.split("RMSE: ")[1].split(" ha")[0]) for x in trial_panel_arith_2]), min([float(x.split("MAD: ")[1].split(" ha")[0]) for x in trial_panel_arith_2]))
evaluate_arithmetic(y_test, y_pred_arith_2_ha, "2 - Normal Veri (Bayesyen Optuna HPO)", best_eq_str_a2)

[PYSR ARITMETIK REGRESYON] - DÜZELTİLMİŞ BAYESYEN OPTUNA HPO & KALICI PANEL
Tamamlanan Deneme : [██████████████████████████] 8 / 8 (%100) | Geçen Süre: 2.60 dk
Anlık En İyi MAE  : 11.1300 ha | RMSE: 41.1900 ha | MAD: 0.5000 ha
-----------------------------------------------------------------------------------------
Deneme 01 -> Kayıp: HuberLoss(0.1 | MaxB: 21 | MAE: 11.26 ha | RMSE: 41.38 ha | MAD: 0.90 ha
Deneme 02 -> Kayıp: HuberLoss(0.3 | MaxB: 22 | MAE: 11.16 ha | RMSE: 41.32 ha | MAD: 1.05 ha
Deneme 03 -> Kayıp: HuberLoss(0.3 | MaxB: 24 | MAE: 11.13 ha | RMSE: 41.40 ha | MAD: 0.50 ha
Deneme 04 -> Kayıp: HuberLoss(0.6 | MaxB: 20 | MAE: 11.13 ha | RMSE: 41.19 ha | MAD: 1.03 ha
Deneme 05 -> Kayıp: HuberLoss(0.3 | MaxB: 25 | MAE: 11.15 ha | RMSE: 41.30 ha | MAD: 0.91 ha
Deneme 06 -> Kayıp: L1DistLoss()  | MaxB: 30 | MAE: 11.19 ha | RMSE: 41.31 ha | MAD: 1.04 ha
Deneme 07 -> Kayıp: HuberLoss(0.1 | MaxB: 23 | MAE: 11.16 ha | RMSE: 41.35 ha | MAD: 0.94 ha
Deneme 08 -> Kayıp: L2DistLoss()

,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),Keşfedilen Aritmetik Formül
0,1 - Normal Veri (Varsayılan Zero-Shot),PySR Arithmetic Regression (Dört İşlem),0.9031,19.4071,110.1527,x2*x2/((x8*x9/x2))
1,2 - Normal Veri (Bayesyen Optuna HPO),PySR Arithmetic Regression (Dört İşlem),0.7571,19.4977,110.2363,x2/(x8 - 1*3.9172635)


(np.float64(0.7571200141858677),
 19.497745651308705,
 np.float64(110.23627672936313))

### Deney 3:

In [12]:
best_eq_str_a3 = ""

try:
    from pysr import PySRRegressor
    model_arith_3 = PySRRegressor(
        niterations=25,
        populations=6,
        population_size=33,
        procs=0,
        multithreading=False,
        binary_operators=["+", "-", "*", "/"],
        unary_operators=[],
        constraints={"/": (-1, 9)},
        loss="L1DistLoss()",
        verbosity=0,
        random_state=42
    ).fit(X_tr_sym.values, y_train_log)
    preds_log_a3_raw = model_arith_3.predict(X_te_sym.values)
    best_eq_str_a3 = str(model_arith_3.sympy())
except Exception:
    poly_a3 = PolynomialFeatures(degree=2, interaction_only=True, include_bias=True)
    model_sym_a3 = LassoCV(cv=3, random_state=42, max_iter=5000).fit(poly_a3.fit_transform(X_tr_sym.values), y_train_log)
    preds_log_a3_raw = model_sym_a3.predict(poly_a3.transform(X_te_sym.values))
    coefs, names = model_sym_a3.coef_, poly_a3.get_feature_names_out(X_tr_sym.columns)
    terms = [f"{coefs[i]:.3f}*{names[i]}" for i in range(len(coefs)) if abs(coefs[i]) > 0.001]
    best_eq_str_a3 = " + ".join(terms[:4]) if terms else f"{model_sym_a3.intercept_:.3f} (sabit)"

y_pred_arith_3_ha = np.clip(np.expm1(np.nan_to_num(preds_log_a3_raw, nan=0.0, posinf=10.0, neginf=0.0)), 0.0, None)

evaluate_arithmetic(y_test, y_pred_arith_3_ha, "3 - Log-Dönüşümlü Veri (Varsayılan Zero-Shot)", best_eq_str_a3)

[PYSR ARITMETIK REGRESYON] - 3 - LOG-DÖNÜŞÜMLÜ VERI (VARSAYILAN ZERO-SHOT) SONUÇLARI
Model -> MAD: 0.6278 ha | MAE: 19.5713 ha | RMSE: 110.2857 ha
Keşfedilen Beyaz-Kutu Denklem: x10/x8
-----------------------------------------------------------------------------------------


,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),Keşfedilen Aritmetik Formül
0,1 - Normal Veri (Varsayılan Zero-Shot),PySR Arithmetic Regression (Dört İşlem),0.9031,19.4071,110.1527,x2*x2/((x8*x9/x2))
1,2 - Normal Veri (Bayesyen Optuna HPO),PySR Arithmetic Regression (Dört İşlem),0.7571,19.4977,110.2363,x2/(x8 - 1*3.9172635)
2,3 - Log-Dönüşümlü Veri (Varsayılan Zero-Shot),PySR Arithmetic Regression (Dört İşlem),0.6278,19.5713,110.2857,x10/x8


(np.float64(0.6277891585058719),
 19.571269958343855,
 np.float64(110.28571197795058))

### Deney 4:

In [13]:
start_time_arith_4 = time.time()
trial_panel_arith_4 = []

def print_live_arith_4(n_total, best_mae, best_rmse, best_mad):
    clear_output(wait=True)
    elapsed_min = (time.time() - start_time_arith_4) / 60.0
    n_done = len(trial_panel_arith_4)
    bar_str = "█" * int(26 * n_done / n_total) + "-" * (26 - int(26 * n_done / n_total))
    print("=========================================================================================")
    print("[PYSR ARITMETIK REGRESYON] - LOG VERI BAYESYEN OPTUNA HPO & KALICI PANEL")
    print("=========================================================================================")
    print(f"Tamamlanan Deneme : [{bar_str}] {n_done} / {n_total} (%{int(n_done/n_total*100)}) | Geçen Süre: {elapsed_min:.2f} dk")
    print(f"Anlık En İyi MAE  : {best_mae:.4f} ha | RMSE: {best_rmse:.4f} ha | MAD: {best_mad:.4f} ha")
    print("-----------------------------------------------------------------------------------------")
    for t_log in trial_panel_arith_4[-8:]:
        print(t_log)
    print("=========================================================================================")

def objective_arith_4(trial):
    loss_choice = trial.suggest_categorical("loss", ["L1DistLoss()", "HuberLoss(0.15)", "HuberLoss(0.3)", "HuberLoss(0.6)", "L2DistLoss()"])
    max_size = trial.suggest_int("maxsize", 20, 30)
    parsimony_val = trial.suggest_float("parsimony", 1e-8, 5e-5, log=True)
    n_iters = trial.suggest_int("niterations", 50, 90)
    n_pops = trial.suggest_int("populations", 8, 14)
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_maes, cv_rmses, cv_mads, cv_scores = [], [], [], []
    
    for tr_idx, val_idx in kf.split(X_tr_sym.values):
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            try:
                from pysr import PySRRegressor
                model_cv = PySRRegressor(
                    niterations=n_iters,
                    populations=n_pops,
                    population_size=35,
                    procs=0,
                    multithreading=False,
                    binary_operators=["+", "-", "*", "/"],
                    unary_operators=[],
                    constraints={"/": (-1, 9), "*": (-1, 9)},
                    loss=loss_choice,
                    maxsize=max_size,
                    parsimony=parsimony_val,
                    verbosity=0,
                    random_state=42
                ).fit(X_tr_sym.values[tr_idx], y_train_log[tr_idx])
                preds_log_raw = model_cv.predict(X_tr_sym.values[val_idx])
            except Exception:
                poly_cv = PolynomialFeatures(degree=2, interaction_only=True, include_bias=True)
                model_cv = LassoCV(cv=3, random_state=42, max_iter=5000).fit(poly_cv.fit_transform(X_tr_sym.values[tr_idx]), y_train_log[tr_idx])
                preds_log_raw = model_cv.predict(poly_cv.transform(X_tr_sym.values[val_idx]))
                
        preds_clean = np.clip(np.expm1(np.nan_to_num(preds_log_raw, nan=0.0, posinf=10.0, neginf=0.0)), 0.0, 10000.0)
        mae_f = mean_absolute_error(y_train[val_idx], preds_clean)
        rmse_f = np.sqrt(mean_squared_error(y_train[val_idx], preds_clean))
        mad_f = np.median(np.abs(y_train[val_idx] - preds_clean))
        
        cv_maes.append(mae_f)
        cv_rmses.append(rmse_f)
        cv_mads.append(mad_f)
        cv_scores.append(0.65 * mae_f + 0.20 * rmse_f + 0.15 * mad_f)
        
    mean_mae, mean_rmse, mean_mad = np.mean(cv_maes), np.mean(cv_rmses), np.mean(cv_mads)
    trial_panel_arith_4.append(f"Deneme {trial.number+1:02d} -> Kayıp: {loss_choice[:13].ljust(13)} | MaxB: {max_size:02d} | MAE: {mean_mae:.2f} ha | RMSE: {mean_rmse:.2f} ha | MAD: {mean_mad:.2f} ha")
    
    print_live_arith_4(8, min([float(x.split("MAE: ")[1].split(" ha")[0]) for x in trial_panel_arith_4]), min([float(x.split("RMSE: ")[1].split(" ha")[0]) for x in trial_panel_arith_4]), min([float(x.split("MAD: ")[1].split(" ha")[0]) for x in trial_panel_arith_4]))
    return np.mean(cv_scores)

study_arith_4 = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_arith_4.optimize(objective_arith_4, n_trials=8, show_progress_bar=False)

best_p_arith_4 = study_arith_4.best_params
best_eq_str_a4 = ""

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    try:
        from pysr import PySRRegressor
        model_arith_4_final = PySRRegressor(
            niterations=120,
            populations=best_p_arith_4["populations"] + 6,
            population_size=40,
            procs=0,
            multithreading=False,
            binary_operators=["+", "-", "*", "/"],
            unary_operators=[],
            constraints={"/": (-1, 9), "*": (-1, 9)},
            loss=best_p_arith_4["loss"],
            maxsize=best_p_arith_4["maxsize"],
            parsimony=best_p_arith_4["parsimony"],
            verbosity=0,
            random_state=42
        ).fit(X_tr_sym.values, y_train_log)
        preds_log_a4_raw = model_arith_4_final.predict(X_te_sym.values)
        best_eq_str_a4 = str(model_arith_4_final.sympy())
    except Exception:
        poly_opt = PolynomialFeatures(degree=2, interaction_only=True, include_bias=True)
        model_sym_opt = LassoCV(cv=3, random_state=42, max_iter=5000).fit(poly_opt.fit_transform(X_tr_sym.values), y_train_log)
        preds_log_a4_raw = model_sym_opt.predict(poly_opt.transform(X_te_sym.values))
        coefs, names = model_sym_opt.coef_, poly_opt.get_feature_names_out(X_tr_sym.columns)
        terms = [f"{coefs[i]:.3f}*{names[i]}" for i in range(len(coefs)) if abs(coefs[i]) > 0.001]
        best_eq_str_a4 = " + ".join(terms[:4]) if terms else f"{model_sym_opt.intercept_:.3f} (sabit)"

y_pred_arith_4_ha = np.clip(np.expm1(np.nan_to_num(preds_log_a4_raw, nan=0.0, posinf=10.0, neginf=0.0)), 0.0, None)

print_live_arith_4(8, min([float(x.split("MAE: ")[1].split(" ha")[0]) for x in trial_panel_arith_4]), min([float(x.split("RMSE: ")[1].split(" ha")[0]) for x in trial_panel_arith_4]), min([float(x.split("MAD: ")[1].split(" ha")[0]) for x in trial_panel_arith_4]))
evaluate_arithmetic(y_test, y_pred_arith_4_ha, "4 - Log-Dönüşümlü Veri (Bayesyen Optuna HPO)", best_eq_str_a4)

[PYSR ARITMETIK REGRESYON] - LOG VERI BAYESYEN OPTUNA HPO & KALICI PANEL
Tamamlanan Deneme : [██████████████████████████] 8 / 8 (%100) | Geçen Süre: 8.77 dk
Anlık En İyi MAE  : 11.3000 ha | RMSE: 40.8800 ha | MAD: 0.9000 ha
-----------------------------------------------------------------------------------------
Deneme 01 -> Kayıp: HuberLoss(0.1 | MaxB: 21 | MAE: 11.63 ha | RMSE: 42.09 ha | MAD: 0.98 ha
Deneme 02 -> Kayıp: HuberLoss(0.3 | MaxB: 22 | MAE: 11.63 ha | RMSE: 42.07 ha | MAD: 1.00 ha
Deneme 03 -> Kayıp: HuberLoss(0.3 | MaxB: 24 | MAE: 11.60 ha | RMSE: 42.11 ha | MAD: 0.90 ha
Deneme 04 -> Kayıp: HuberLoss(0.6 | MaxB: 20 | MAE: 11.61 ha | RMSE: 42.02 ha | MAD: 1.04 ha
Deneme 05 -> Kayıp: HuberLoss(0.3 | MaxB: 25 | MAE: 11.79 ha | RMSE: 42.14 ha | MAD: 1.10 ha
Deneme 06 -> Kayıp: L1DistLoss()  | MaxB: 30 | MAE: 11.63 ha | RMSE: 42.08 ha | MAD: 0.99 ha
Deneme 07 -> Kayıp: HuberLoss(0.1 | MaxB: 23 | MAE: 11.63 ha | RMSE: 42.09 ha | MAD: 0.98 ha
Deneme 08 -> Kayıp: L2DistLoss()  |

,Deney Aşaması,Model Motoru,MAD (ha),MAE (ha),RMSE (ha),Keşfedilen Aritmetik Formül
0,1 - Normal Veri (Varsayılan Zero-Shot),PySR Arithmetic Regression (Dört İşlem),0.9031,19.4071,110.1527,x2*x2/((x8*x9/x2))
1,2 - Normal Veri (Bayesyen Optuna HPO),PySR Arithmetic Regression (Dört İşlem),0.7571,19.4977,110.2363,x2/(x8 - 1*3.9172635)
2,3 - Log-Dönüşümlü Veri (Varsayılan Zero-Shot),PySR Arithmetic Regression (Dört İşlem),0.6278,19.5713,110.2857,x10/x8
3,4 - Log-Dönüşümlü Veri (Bayesyen Optuna HPO),PySR Arithmetic Regression (Dört İşlem),2.3143,19.7942,109.9314,(-x0 - x2 - 6.161354)*(-0.06144468)


(np.float64(2.3143082640835857),
 19.79417025380598,
 np.float64(109.93142465786511))